In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import mimetypes
import os
import re
import sqlite3
import time
import unicodedata
from collections import Counter
from operator import itemgetter
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import requests
from more_itertools import flatten, unique_everseen
from rapidfuzz import fuzz, process

from aymurai.llm_providers import OllamaLLMProvider
from aymurai.meta.entities import CanonicalEntities, CanonicalEntity
from aymurai.utils.json_data import get_pretty, load_json, save_json

In [ ]:
API_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://localhost:8999")
ENDPOINT = f"{API_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "/resources/data/restricted/disambiguation-eval")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    """
    Calls the document extraction API with the given file.

    Args:
        session (requests.Session): The HTTP session to use for the request.
        file_path (Path): The path to the file to be extracted.
    Returns:
        dict[str, object]: The response payload containing status and details.
    """
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")
documents

## Fetch data from database

In [ ]:
# Anonymization document
conn = sqlite3.connect("/workspace/sqlite/database.db")
anonymization_document = pd.read_sql_query("SELECT * FROM anonymization_document", conn)
conn.close()

anonymization_document

In [ ]:
# Anonymization document paragraphs
conn = sqlite3.connect("/workspace/sqlite/database.db")
anonymization_document_paragraph = pd.read_sql_query(
    "SELECT * FROM anonymization_document_paragraph", conn
)
conn.close()

anonymization_document_paragraph

In [ ]:
# Anonymization paragraph
conn = sqlite3.connect("/workspace/sqlite/database.db")
anonymization_paragraph = pd.read_sql_query(
    "SELECT * FROM anonymization_paragraph", conn
)
conn.close()

anonymization_paragraph

## Get particular document

In [ ]:
doc_filename = "<DOC_FILENAME>"  # Replace with actual document filename

# Get document ID
document_id = anonymization_document.loc[
    anonymization_document["name"].str.startswith(doc_filename), "id"
].values[0]
document_id

In [ ]:
# Get paragraph IDs for the document
paragraph_ids = anonymization_document_paragraph.loc[
    anonymization_document_paragraph["document_id"] == document_id, "paragraph_id"
].tolist()

paragraph_ids

In [ ]:
# Get paragraphs
doc_paragraphs = anonymization_paragraph.loc[
    anonymization_paragraph["id"].isin(paragraph_ids)
]
doc_paragraphs

In [ ]:
# Get validations
validations = doc_paragraphs.loc[:, "validation"].map(eval).tolist()
validations = list(flatten(validations))
validations

In [ ]:
# Parse validations to extract labels and alt texts
validations = [
    {
        "aymurai_label": validation["attrs"]["aymurai_label"],
        "text": validation["attrs"].get("aymurai_alt_text") or validation["text"],
    }
    for validation in validations
]

validations

In [ ]:
# Remove duplicates and sort
validations = list(unique_everseen(validations))
validations = sorted(
    validations,
    key=itemgetter("aymurai_label", "text"),
)
validations

In [ ]:
# Fix 'TEL' labels
for validation in validations:
    if validation["aymurai_label"] == "TEL":
        validation["aymurai_label"] = "TELEFONO"

validations

In [ ]:
# Map 'TEXTO_ANONIMIZAR' to 'NOMBRE_ARCHIVO' when applicable
for validation in validations:
    extension = os.path.splitext(validation["text"])[1].lower()
    if validation["aymurai_label"] == "TEXTO_ANONIMIZAR":
        if extension.endswith(
            (
                "jpg",
                "jpeg",
                "png",
                "pdf",
                "docx",
                "odt",
                "mp4",
                "enc",
            )
        ):
            validation["aymurai_label"] = "NOMBRE_ARCHIVO"

validations

## Cluster entities based on textual

In [ ]:
def normalize(s: str) -> str:
    """
    Normalize string for clustering.

    Args:
        s (str): input string

    Returns:
        str: normalized string
    """
    # strip accents, lowercase, collapse spaces
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
    s = " ".join(s.lower().split())
    return s


def cluster_with_cdist(
    entities: list[str], threshold: int = 90, scorer: callable = fuzz.token_set_ratio
):
    """
    Cluster entities based on similarity using a distance matrix.

    Args:
        entities (list[str]): list of entity strings
        threshold (int): similarity threshold for clustering
        scorer (callable): similarity scoring function

    Returns:
        list[list[tuple[str, str]]]: clusters of (original, normalized) entity tuples
    """
    # matrix of similarities on normalized strings
    # normalize however you like; here just lower + strip
    normed = [" ".join(e.lower().split()) for e in entities]
    sim = process.cdist(normed, normed, scorer=scorer, score_cutoff=threshold)
    sim = np.array(sim)

    # union-find
    parent = list(range(len(normed)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri

    # link pairs above threshold (upper triangle only)
    n = len(normed)
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= threshold:
                union(i, j)

    # collect clusters
    clusters = {}
    for idx in range(n):
        root = find(idx)
        clusters.setdefault(root, []).append((entities[idx], normed[idx]))
    return list(clusters.values())


def pick_canonical(cluster: list[tuple[str, str]]) -> str:
    """
    Pick canonical text from a cluster.

    Args:
        cluster (list[tuple[str, str]]): cluster of (original, normalized) entity tuples

    Returns:
        str: chosen canonical text
    """
    # prefer longest original; fall back to first
    return max(cluster, key=lambda x: len(x[0]))[0]


In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in validations],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
def parse_item(item: tuple[str, ...]) -> tuple[str, str, str]:
    """
    Parse an item into (label, orig, norm).

    Accepts:
      - (orig, norm, label)
      - (labelled_orig, labelled_norm) with prefix 'LABEL:'
    Args:
        item (tuple[str, ...]): input item

    Returns:
        tuple[str, str, str]: (label, orig, norm)
    """
    if len(item) == 3:
        orig, norm, label = item
        return label, orig, norm

    # len == 2: assume "LABEL:text"
    labelled_orig, labelled_norm = item
    label, orig = labelled_orig.split(":", 1)
    _, norm = labelled_norm.split(":", 1)
    return label, orig, norm


def pick_cluster_label(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Pick the most common label from parsed items.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen label
    """
    labels = [lbl for lbl, _, _ in parsed_items]
    # majority vote; fallback to first
    return Counter(labels).most_common(1)[0][0]


def pick_canonical_text(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Choose the longest original surface form; tweak as needed.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen canonical text
    """
    return max(parsed_items, key=lambda x: len(x[1]))[1]


def clusters_to_canonical_entities(
    clusters: list[list[tuple[str, str]]],
) -> list[CanonicalEntity]:
    """
    Convert clusters to CanonicalEntity objects.

    Args:
        clusters (list[list[tuple[str, str]]]): clusters of (original, normalized) entity tuples

    Returns:
        list[CanonicalEntity]: list of CanonicalEntity objects
    """
    canonical_entities = []

    for cluster in clusters:
        parsed = [parse_item(item) for item in cluster]  # [(label, orig, norm), ...]
        label = pick_cluster_label(parsed)
        canonical_text = pick_canonical_text(parsed)
        aliases = sorted({orig for _, orig, _ in parsed})
        ce = CanonicalEntity(
            aymurai_label=label,
            canonical_text=canonical_text,
            aliases=aliases,
            attributes={},
            relations=[],
        )
        canonical_entities.append(ce)

    return canonical_entities

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)
canonical_entities

In [ ]:
# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(doc_filename))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

# Save canonical entities to JSON for reviewing
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "/resources/data/restricted/disambiguation-eval/canonical-entities/REVIEWING/"
    + target_filename
    + "-canonical-entities.json",
)

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "/resources/data/restricted/disambiguation-eval/canonical-entities/REVIEWING/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]
canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Persist reviewed canonical entities
save_json(
    canonical_entities,
    "/resources/data/restricted/disambiguation-eval/canonical-entities/"
    + target_filename
    + "-canonical-entities.json",
)

## CanonicalEntity extraction

In [ ]:
# Available models
model = "phi4:14b"

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=model)
provider.generate("hola, ¿cómo estás?")

In [ ]:
system_prompt = """
Eres un asistente especializado en anonimización de sentencias judiciales.
Tu tarea es agrupar las menciones de entidades nombradas detectadas por un modelo de NER en **entidades canónicas** sin omitir ninguna mención válida.
Una **entidad canónica** es la representación única de una entidad real.
Agrupa todas las menciones textuales (aliases) que se refieren a una misma persona, documento, lugar, número, fecha, etc., aunque aparezcan con variantes ortográficas o abreviadas.

# Reglas
- Usa solo información presente en el documento y en las menciones del NER; no inventes datos ni concluyas hechos no expresos.
- Toda mención listada por el NER debe evaluarse. Si representa una entidad real, inclúyela en alguna entidad canónica.
- Puede haber falsos positivos y/o negativos, menciones ambiguas o incompletas, por lo que debes evaluar cada mención cuidadosamente.
- Cada entidad canónica debe incluir:
  - `aymurai_label` (etiqueta del NER, p. ej. "PER", "DNI", etc.).
  - `canonical_text` (forma normalizada: nombres completos, fechas normalizadas, etc.).
  - `aliases` (todas las menciones textuales relevantes).
  - `attributes` (diccionario opcional con roles u otras notas, p. ej. {"role": "Juez/a"}).

# Notas
- `aliases` debe conservar las menciones textuales tal como aparecen en el documento.
- `attributes` es especialmente útil para distinguir roles procesales. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"

# Ejemplo
Q: Con fecha 3 de marzo de 2023, la Sra. Laura Beatriz Gómez, DNI 42.987.654, con domicilio en calle Falsa 123, Barrio Los Pinos, Villa Azul, denunció a su expareja, el Sr. Martín Alberto Rodríguez, DNI 27.654.321, con domicilio en calle Real 789, por hechos de violencia física, psicológica y amenazas con armas blancas.
A: ```json
[
  {
    "aymurai_label": "PER",
    "canonical_text": "Laura Beatriz Gómez",
    "aliases": ["Laura Beatriz Gómez"],
    "attributes": {"role": "Denunciante"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "42987654",
    "aliases": ["42.987.654"]
  },
  {
    "aymurai_label": "DIRECCION",
    "canonical_text": "calle Falsa 123",
    "aliases": ["calle Falsa 123"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "Barrio Los Pinos, Villa Azul",
    "aliases": ["Barrio Los Pinos, Villa Azul"]
  },
  {
    "aymurai_label": "PER",
    "canonical_text": "Martín Alberto Rodríguez",
    "aliases": ["Martín Alberto Rodríguez"],
    "attributes": {"role": "Denunciado"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "27654321",
    "aliases": ["27.654.321"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "calle Real 789",
    "aliases": ["calle Real 789"]
  }
]
```
"""

In [ ]:
user_prompt_template = """
A continuación se proporciona un documento judicial y las menciones de entidades nombradas detectadas por un modelo de NER.

# Documento
{document_text}

# Menciones de entidades canónicas agrupadas
{canonical_entities}

# Instrucciones
1. Evalúa cada entidad canónica listada, poniendo especial atención en las aliases.
2. Si alguna alias no corresponde a esa entidad, elimínala de la lista de aliases y créala como una nueva entidad canónica.
3. Las diferencias textuales correctas en los aliases pueden ser variantes ortográficas, abreviaciones, iniciales o errores tipográficos.
4. Diferencias sutiles que permiten desambiguar entidades distintas (por ejemplo, nombres similares de personas diferentes, fechas próximas pero distintas, números de documentos, códigos de identificación o nombres de archivos con pequeñas variaciones, etc.).
5. Normaliza canonical_text; mantén los aliases tal como aparecen.
6. Identifica roles u otros atributos relevantes en el campo attributes (por ejemplo, {{"role": "Denunciante"}}).
"""

In [ ]:
# Extract document
session = requests.Session()
doc_path = next((doc for doc in documents if doc.name == doc_filename), None)
document = call_extraction_api(session, Path(doc_path))
document = document.get("detail", {}).get("document")

if not document:
    raise ValueError("Document text is empty or not found.")

In [ ]:
# Remove empty 'entity_id', 'relations' and 'attributes' fields
canonical_entities = [
    {
        k: v
        for k, v in ce.items()
        if k not in ("entity_id", "relations", "attributes") or v
    }
    for ce in canonical_entities
]

# Prepare user prompt
user_prompt = user_prompt_template.format(
    document_text="\n".join(document).strip(),
    canonical_entities=get_pretty(canonical_entities),
)

print(user_prompt)

In [ ]:
# Get canonical entities from the model
provider = OllamaLLMProvider(model=model)
response = provider.generate(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    options={"temperature": 0, "num_ctx": 16_384},
    format=CanonicalEntities.model_json_schema(),
)

In [ ]:
response